# Reproducing Fedele et al. (2017) from CRCNS ieeg-1

Paper. Fedele T, Burnos S, Boran E, Krayenbühl N, Hilfiker P, Grunwald T, Sarnthein J (2017).
Resection of high frequency oscillations predicts seizure outcome in the individual patient.
*Scientific Reports* 7, 13836. [doi:10.1038/s41598-017-13064-1](https://doi.org/10.1038/s41598-017-13064-1).
The claims file (`claims/fedele-2017.yaml`) was written from the version of record (PMC5653833) and
its supplement (Table 1S).

Data set. CRCNS ieeg-1, [doi:10.6080/K06Q1VD5](https://doi.org/10.6080/K06Q1VD5), read from the
public bucket `crcnsarchive` under `ieeg-1/live/`. CRCNS data sets are not versioned, so the version
is recorded as `unversioned`; the paper cites the data set's DOI, which names this one deposit, and
there is no other version to choose. To make a later change detectable, the notebook computes the
MD5 of every patient file as it streams it and compares it with the data set's `checksums.md5`
(pat_01 4f4b6cd9e48f7da959785f08b6358b8d, pat_02 892385a67a24b25eaaeb4e53e1b46483,
pat_03 decf88fe2b3d2ed89d8730c57ad600bd, pat_04 ee66253f637f4173cb4ec0a14aff90a3,
pat_05 86b8324f46985bedacd5fa665fc0b9f5, pat_06 673073e5b7b7823c1627f38e905926a4,
pat_07 41435ba7a9fbef08cf2122486931f205, pat_08 9cec13f294c33f8c8b5b6cd94818a68e,
pat_09 3e463925f07508c66e97f4bcc316e50a, pat_10 7e517b8d070897c2fc64ee9f446a84ab,
pat_11 ba2f3677e4fdb97b60f69657208271c1, pat_12 63a7da3208ea30e1661cd29bc89582a4,
pat_13 e304540b9bad242f313c42c77214690e, pat_14 504dad9455c27a0ffe6e38e05a31c565,
pat_15 f23ed5f8fa6ba9cbdddf937cb2e5d27b, pat_16 048dc646f120cdca491b5a0067e63e82,
pat_17 bbf09498562bac0120274292f0f777bf, pat_18 ffdc1d8ed2f295eff5bfb5189c24cb79,
pat_19 f7b3fb81dff0f842308c26df6f92f166, pat_20 4e4e9517e9e0b1abeddda7a5ed03b8c8; scripts.zip
6421f53aa2a2df1c0797afd7f8877f8c).

What the data set holds. One tar.gz per patient (92.5 GB in all) with one MATLAB file per
interval of slow-wave sleep: the raw iEEG with its labels, the bipolar montage, and the detected
ripples (R), fast ripples (FR) and FRandR as start and end samples per bipolar channel. It holds no
seizure outcome, no follow-up, no epilepsy type, no resected channels and no seizure onset zone.
These come from the paper: the outcome, follow-up and epilepsy type from Table 1 (fetched from the
Europe PMC full text) and the resected channels and HFO areas from Table 1S (the supplement PDF,
fetched from the publisher and checked by MD5, its cell colours read from the PDF's vector fills).

Streaming. A gzip stream cannot be read at an offset, so every file is streamed once and reduced
as it loads (`ieeg1.scan_patient`): about 15 minutes for the 92.5 GB with four workers, each holding
one interval (at most about 2 GB). Nothing is written to disk.

| Claim | Tested |
|---|---|
| table-s1-link (link) | tested |
| cohort | tested |
| table1-nights-intervals | tested |
| hfo-amplitudes | tested |
| followup-not-different | tested |
| table-s1-rates | tested |
| fig1-patient4 | tested |
| frandr-area-resection-per-patient | tested |
| ripple-fr-area-resection-per-patient | tested |
| table2-frandr-prediction | tested |
| table2-ripple-fr-prediction | tested |
| frandr-more-specific | tested |
| engel-prediction | tested |
| ete-soz-not-resected | blocked: the seizure onset zone is in neither the data set nor the paper |
| fig2-patient17 | tested (the SOZ part is not) |
| fig3-patient1 | tested |
| test-retest-group | tested |
| test-retest-by-group | tested |
| spectral-classes | tested |
| spectral-tle-ete-no-difference | tested |
| tp-patients-hfo-area-location | blocked: the electrode schemes do not give the region of each contact |


In [ ]:
import time
import warnings
from multiprocessing import get_context

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ieeg1

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
scans = ieeg1.scan_all(workers=4)       # streams all 20 patients once (about 92 GB)
print(f"streamed {sum(s['bytes'] for s in scans.values()) / 1e9:.1f} GB from CRCNS {ieeg1.DATASET} "
      f"({ieeg1.VERSION}, doi:{ieeg1.DATASET_DOI}) in {(time.time() - started) / 60:.1f} min")
s1 = ieeg1.load_table_s1()              # the paper's Table 1S (supplement), colours included
t1 = pd.DataFrame(ieeg1.load_table1())  # the paper's Table 1 (Europe PMC full text)
t1["Patient"] = t1["Patient"].astype(int)
t1 = t1.set_index("Patient")
PIDS = sorted(scans)

## Data check

Each check of the description against the values is stated after the outputs.

In [ ]:
md5 = ieeg1.checksums()
rows = []
fingerprints = {}
for p in PIDS:
    sc = scans[p]
    ivs = sc["intervals"]
    nights = pd.Series([iv["night"] for iv in ivs]).value_counts().sort_index()
    dur = np.array([iv["n_samples"] / iv["fs"] for iv in ivs])
    for iv in ivs:
        fingerprints.setdefault(iv["x_fingerprint"], []).append((p, iv["night"], iv["interval"]))
    rows.append({
        "patient": p, "md5_ok": sc["md5"] == md5[f"data/pat_{p:02d}.tar.gz"],
        "members": len(sc["members"]), "non_mat": sum(not m[0].endswith(".mat") for m in sc["members"]),
        "patient_field_ok": all(iv["patient"] == p for iv in ivs),
        "nights": len(nights), "intervals": len(ivs), "max_per_night": int(nights.max()),
        "interval_index_ok": sorted(iv["interval"] for iv in ivs) == list(range(1, len(ivs) + 1)),
        "dur_min_s": dur.min(), "dur_max_s": dur.max(), "n_not_300s": int(np.sum(dur != 300)),
        "fs": sorted({iv["fs"] for iv in ivs}), "nch": sorted({iv["nch"] for iv in ivs}),
        "nch_eq_rows": all(iv["nch"] == iv["n_rows_x"] for iv in ivs),
        "n_bipolar": len(ivs[0]["labels_bip"]),
        "labels_same_all": len({tuple(iv["labels_bip"]) for iv in ivs}) == 1
                           and len({tuple(iv["labels"]) for iv in ivs}) == 1,
        "x_min": min(iv["x_range"][0] for iv in ivs), "x_max": max(iv["x_range"][1] for iv in ivs),
        "bip_median_abs": np.median([iv["bip_median_abs"] for iv in ivs]),
        "R": sum(len(iv["R"]) for iv in ivs), "FR": sum(len(iv["FR"]) for iv in ivs),
        "FRandR": sum(len(iv["FRandR"]) for iv in ivs),
        "ev_out_of_range": sum(iv[k + "_out_of_range"] for iv in ivs for k in ieeg1.TYPES),
        "ev_noninteger": sum(iv[k + "_nonint"] for iv in ivs for k in ieeg1.TYPES),
    })
info = pd.DataFrame(rows).set_index("patient")
print(info.round(1).to_string())
print("\nAll 20 files match checksums.md5:", bool(info.md5_ok.all()),
      "; files other than .mat in the archives:", int(info.non_mat.sum()))
print("Intervals in total:", int(info.intervals.sum()), "; nights in total:", int(info.nights.sum()))
print("Interval durations (s): min", info.dur_min_s.min(), "max", info.dur_max_s.max(),
      "; intervals not 300 s long:", int(info.n_not_300s.sum()), "of", int(info.intervals.sum()))
dups = {k: v for k, v in fingerprints.items() if len(v) > 1}
print("Intervals whose first 10 s of raw signal are identical to another interval's:", len(dups))
print("Sampling rates found:", sorted({f for fs in info.fs for f in fs}))

In [ ]:
# Event tables: which events fall outside the signal, and what the samples look like.
oor = []
for p in PIDS:
    for iv in scans[p]["intervals"]:
        n = iv["n_samples"]
        for k in ieeg1.TYPES:
            ev = iv[k]
            bad = (ev[:, 2] >= n) | (ev[:, 1] < 0)
            if bad.any():
                oor.append({"patient": p, "type": k, "n": int(bad.sum()),
                            "max_end_minus_length": int((ev[bad, 2] - (n - 1)).max())})
oor = pd.DataFrame(oor)
print("Events whose end lies beyond the last sample (0-based end - last index):")
print(oor.groupby("type").agg(n=("n", "sum"), max_over=("max_end_minus_length", "max")).to_string())
allev = {k: np.concatenate([iv[k] for p in PIDS for iv in scans[p]["intervals"]]) for k in ieeg1.TYPES}
for k, ev in allev.items():
    d = (ev[:, 2] - ev[:, 1] + 1) / ieeg1.FS * 1000
    print(f"{k}: {len(ev)} events, duration ms 5/50/95th percentile {np.percentile(d, [5, 50, 95]).round(1)}, "
          f"end before start {int(np.sum(ev[:, 2] < ev[:, 1]))}")
# Is every FRandR event also an R event and an FR event with the same start and end?
same = {"R": 0, "FR": 0}
total = 0
for p in PIDS:
    for iv in scans[p]["intervals"]:
        for k in ("R", "FR"):
            a = {tuple(r) for r in iv[k]}
            same[k] += sum(tuple(r) in a for r in iv["FRandR"])
        total += len(iv["FRandR"])
print(f"FRandR events that are also R events with the same channel, start and end: {same['R']} of {total}; "
      f"also FR events: {same['FR']} of {total}")

In [ ]:
# Timing: band power inside the events at the marked times against the same windows 1 s later.
tim = []
for p in PIDS:
    ivs = scans[p]["intervals"]
    for k in ("R", "FR"):
        a = np.array([iv["timing_" + k] for iv in ivs], float)
        tim.append({"patient": p, "type": k, "median_power_marked": np.nanmedian(a[:, 0]),
                    "median_power_shifted_1s": np.nanmedian(a[:, 1]),
                    "frac_events_marked_higher": np.nanmedian(a[:, 2]),
                    "intervals_frac_below_0.6": int(np.sum(a[:, 2] < 0.6))})
tim = pd.DataFrame(tim)
print(tim.round(3).to_string(index=False))
print("\nMedian over patients of the fraction of events with more band power at the marked time:",
      tim.groupby("type").frac_events_marked_higher.median().round(3).to_dict())

In [ ]:
# Table 1S of the paper against the deposit's bipolar labels.
link = []
for p in PIDS:
    dep = [ieeg1.label_key(l) for l in scans[p]["intervals"][0]["labels_bip"]]
    t = s1[p]
    rated = [c for c in t["channels"] if c in t["rates"]["FRandR"]]
    link.append({"patient": p, "S1_channels": len(t["channels"]), "S1_with_rates": len(rated),
                 "S1_gray": len(t["excluded"]), "S1_red": len(t["resected"]),
                 "S1_rated_in_deposit": sum(c in dep for c in rated),
                 "S1_gray_in_deposit": sum(c in dep for c in t["excluded"]),
                 "deposit_not_in_S1": [c for c in dep if c not in t["channels"]],
                 "S1_ILAE": t["ilae"], "T1_ILAE": int(t1.loc[p, "Outcome (ILAE)"])})
link = pd.DataFrame(link).set_index("patient")
print(link.to_string())
print("\nRated channels of Table 1S found in the deposit:", int(link.S1_rated_in_deposit.sum()), "of",
      int(link.S1_with_rates.sum()))
print("ILAE class of Table 1S equals Table 1 for", int((link.S1_ILAE == link.T1_ILAE).sum()), "of 20 patients")
print("Patient 18: the PML contacts, which Table 1S writes both as PML and as PM, are spelt PM on both sides (ieeg1.label_key).")

Checks.

- All 20 files match `checksums.md5`; each holds only `.mat` files, one per interval, and the
  `patient_number` field equals the file's patient. 385 intervals over 58 nights; no two intervals
  share their first 10 s of raw signal, so no recording is deposited twice.
- The data description says each file holds 5 minutes. The data disagree: 93 of 385 intervals are
  not 300 s long (204 s to 720 s). Rates are therefore computed on each interval's actual length;
  5 minutes per interval is shown as a variant (claim table-s1-rates).
- Sampling rate 2000 Hz in every file, as described. `nch` equals the rows of `x`. Labels are the
  same in every interval of a patient.
- The unit of `x` is not stated in the data description. The median absolute bipolar signal is
  20 to 93 per patient, and the amplitudes of claim hfo-amplitudes match the paper's microvolts, so
  the values are consistent with microvolts.
- Event samples are 1-based (the data set's view_data.m indexes with them directly); some are not
  whole numbers and are rounded, as view_data.m does for FR and FRandR. 745 events end after the last
  sample of their interval, by up to 926 samples; their amplitudes are taken over the part inside.
- Every FRandR event is also an R event and an FR event with the same channel, start and end: in
  the deposit an FRandR event is the detector's merged event (the detector code in detector.zip,
  RUN_MorphologyDetector.m, keeps events with mark 3 in all three lists).
- Clock: the band power of the filtered signal inside the marked events is higher than in the same
  windows shifted by 1 s for a median of 94% of ripples and 98.5% of FR per patient, with no interval
  below 60% for either type, so the event tables share the clock of the signal in every patient.
- Table 1S against the deposit: every rated channel of Table 1S is found among the deposit's bipolar
  labels. Patient 18's PML contacts are written both as PML and as PM in the table; the spelling is
  unified. Table 1S lists under patient 11 fifteen grayed channels (IAR3-4 to PMR15-16) that are not
  among patient 11's channels in the deposit but are exactly the 15 deposit channels of patient 14
  that the table does not list: the gray list appears to be placed one patient too early. Patient 19
  has 4 deposit channels (PL1-2, PL9-10, PL17-18, PL25-26) that Table 1S does not list.
- ILAE classes of Table 1S and Table 1 agree for all 20 patients.

### Rates, channel sets and HFO areas used by several claims

Rate of a channel: events per minute (Table 1S; Fig. 3A). Pooling over intervals: readings (a) all
events over the total recorded time, (b) the mean of per-interval rates, (c) the mean of per-night
means (claims file, opening comment). Channel set (a): the channels with rates in Table 1S that are
not grayed; (b): all bipolar channels of the deposit. HFO area: rate above the 95th percentile by
Matlab prctile (Methods, "Definition of the HFO area by rate thresholding"), strictly greater as
the main reading. Claim table-s1-rates decides the pooling used elsewhere.

In [ ]:
# Rates per channel, pooled over intervals, for every patient and HFO type, and the channel sets.
counts = {(p, k): ieeg1.count_matrix(scans[p], k) for p in PIDS for k in ieeg1.TYPES}
labels = {p: [ieeg1.label_key(l) for l in scans[p]["intervals"][0]["labels_bip"]] for p in PIDS}
chanset = {}
for p in PIDS:
    t = s1[p]
    chanset[(p, "a")] = [i for i, l in enumerate(labels[p]) if l in t["rates"]["FRandR"] and l not in t["excluded"]]
    chanset[(p, "b")] = list(range(len(labels[p])))


def rates(p, k, reading="a"):
    return ieeg1.pooled_rates(*counts[(p, k)], reading=reading)


def area(p, k, reading="a", cs="a", strict=True):
    keep = chanset[(p, cs)]
    idx, thr = ieeg1.hfo_area(rates(p, k, reading)[keep], strict)
    return {labels[p][keep[i]] for i in idx}, thr


print("Channels per patient in set (a), not grayed in Table 1S:", {p: len(chanset[(p, "a")]) for p in PIDS})

## table-s1-link (link)

> The deposit's patients, nights and intervals are those of the paper's Table 1, and its bipolar
> channels are those of Table 1S, whose rates they reproduce.

This link claim is added by the author (protocol, Author, step 1.4): the resected channels and the
outcomes that most claims need come from the paper's Table 1S and Table 1, matched to the deposit by
patient number and bipolar label.

In [ ]:
# Link: the deposit's patients and channels matched to the paper's Table 1 and Table 1S.
print("Patients in the deposit:", PIDS, "(patient_number field equals the file name for all:",
      bool(info.patient_field_ok.all()), ")")
cmp = pd.DataFrame({"T1_nights": t1["Nights"].astype(int), "dep_nights": info.nights,
                    "T1_intervals": t1["Intervals"].astype(int), "dep_intervals": info.intervals})
print("Patients whose nights and intervals both equal Table 1:",
      int(((cmp.T1_nights == cmp.dep_nights) & (cmp.T1_intervals == cmp.dep_intervals)).sum()), "of 20")
matched = int(link.S1_rated_in_deposit.sum())
print(f"Channels of Table 1S (with rates) matched by bipolar label: {matched} of {int(link.S1_with_rates.sum())} "
      f"({matched / link.S1_with_rates.sum():.1%})")
ok = n = 0
for p in PIDS:
    for k in ieeg1.TYPES:
        r = rates(p, k, "a")
        for c, v in s1[p]["rates"][k].items():
            if c in labels[p]:
                n += 1
                ok += abs(r[labels[p].index(c)] - v) <= max(0.1 * v, 0.1)
print(f"Matched channel and type rates within 10% or 0.1/min of Table 1S (pooling a): {ok} of {n} ({ok / n:.1%})")
print("Gray (excluded) channels of Table 1S found in the deposit:", int(link.S1_gray_in_deposit.sum()), "of",
      int(link.S1_gray.sum()))
print("Patient 11's gray channels in Table 1S:", sorted(s1[11]["excluded"]))
print("Patient 14's deposit labels not listed in Table 1S:", link.loc[14, "deposit_not_in_S1"])

**Result: reproduced.** The deposit holds the 20 patients of Table 1, numbered as there, with
the nights and intervals of Table 1 for all 20. All 600 rated channels of Table 1S are found among
the deposit's bipolar labels (100%), and 1766 of 1800 channel and type rates agree with Table 1S
within 10% or 0.1/min (98.1%). The gray (excluded) channels match for 22 of 37: the 15 listed under
patient 11 belong to patient 14 in the deposit.

## cohort

> The study included 20 patients, 9 with mesial temporal lobe epilepsy and 11 with extratemporal
> epilepsy; 13 became seizure free (ILAE 1) and 7 had recurrent seizures, a seizure-free rate of 65%.

In [ ]:
tle = t1["Epilepsy"] == "TLE"
ilae = t1["Outcome (ILAE)"].astype(int)
print("Patients in the deposit:", len(PIDS))
print("Table 1: TLE", int(tle.sum()), ", ETE", int((~tle).sum()), "; ILAE 1", int((ilae == 1).sum()),
      ", ILAE 2-6", int((ilae > 1).sum()))
sf = int((ilae == 1).sum())
ci = stats.binomtest(sf, 20).proportion_ci(method="exact")
print(f"Seizure-free rate {sf / 20:.0%}, Clopper-Pearson 95% CI {ci.low:.1%}-{ci.high:.1%}")
print("ILAE in Table 1S equals Table 1 for", int((link.S1_ILAE == link.T1_ILAE).sum()), "of 20")
print("\nWhat the deposit itself says about each patient: electrode name prefixes of the bipolar channels")
import re as _re
pref = {p: sorted({_re.match(r"[A-Za-z]+", l).group(0) for l in labels[p]}) for p in PIDS}
for p in PIDS:
    print(p, t1.loc[p, "Epilepsy"], "ILAE", ilae[p], pref[p])
mtl = {"A", "AL", "AR", "AHL", "AHR", "PHL", "PHR", "ECL", "ECR", "EL", "ER", "HL", "HR", "PL", "PR"}
only_mtl = [p for p in PIDS if set(pref[p]) <= mtl]
print("Patients whose channels are all named after mesial temporal depth electrodes:", only_mtl)

**Result: partially reproduced.** The deposit holds 20 patients. Their epilepsy type and seizure
outcome are not in the deposit: by Table 1 there are 9 TLE and 11 ETE patients and 13 seizure free
(65%, Clopper-Pearson 40.8-84.6%), and Table 1S gives the same ILAE class for all 20. From the
deposit alone, the channel names of patients 2 to 9 are all of mesial temporal depth electrodes,
and patient 1 adds IAR and IPR strips, which is consistent with Table 1's 9 TLE patients but does
not prove it; the outcome cannot be read from the deposit.

## table1-nights-intervals

> The number of recorded nights and of 5-minute intervals varies across patients as listed in
> Table 1.

In [ ]:
cmp["match"] = (cmp.T1_nights == cmp.dep_nights) & (cmp.T1_intervals == cmp.dep_intervals)
cmp["max_intervals_per_night"] = info.max_per_night
print(cmp.to_string())
print("Patients matching Table 1 on both:", int(cmp.match.sum()), "of 20")
print("Largest number of intervals in one night:", int(info.max_per_night.max()),
      "(patient", int(info.max_per_night.idxmax()), ")")
print("Patients with more than six intervals in some night:", list(info.index[info.max_per_night > 6]))

**Result: reproduced.** All 20 patients have the nights and intervals of Table 1. The deposit
also shows what Table 1 implies: 14 patients have more than six intervals in some night, up to 16
(patient 5), against the Methods' "up to six intervals" per night.

## hfo-amplitudes

> Detected ripples had median amplitude 27.4 microvolt peak-to-peak (interquartile range 15.0) and FR
> median amplitude 9.2 microvolt peak-to-peak (interquartile range 7.5).

Amplitude = maximum minus minimum inside the event, on (a) the band-passed signal (the data set's
FIR_2kHz.mat filters, forward and backward as in its view_data.m) or (b) the unfiltered bipolar
signal; pooled over (i) all channels or (ii) the channels not grayed in Table 1S.

In [ ]:
def iqr(v):
    q = np.percentile(v, [25, 75])
    return q[1] - q[0]


amp_rows = []
for k in ("R", "FR"):
    for j, sig in enumerate(["(a) band-passed", "(b) unfiltered"]):
        for pool in ["(i) all channels", "(ii) channels not grayed"]:
            vals = []
            for p in PIDS:
                for iv in scans[p]["intervals"]:
                    a = iv["amp_" + k][:, j]
                    ch = iv[k][:, 0]
                    keep = ~np.isnan(a)
                    if pool.startswith("(ii)"):
                        ok_ch = np.array([0 <= c < len(labels[p]) and labels[p][c] in s1[p]["rates"]["FRandR"]
                                          and labels[p][c] not in s1[p]["excluded"] for c in ch], bool)
                        keep &= ok_ch
                    vals.append(a[keep])
            v = np.concatenate(vals)
            amp_rows.append({"type": k, "signal": sig, "pooling": pool, "events": len(v),
                             "median_uVpp": np.median(v), "IQR_uVpp": iqr(v), "min": v.min(), "max": v.max()})
amp = pd.DataFrame(amp_rows)
print(amp.round(2).to_string(index=False))
print("\nPaper: ripples median 27.4 (IQR 15.0), FR median 9.2 (IQR 7.5) uVpp; tolerance 10%.")
for sig in amp.signal.unique():
    for pool in amp.pooling.unique():
        a = amp[(amp.signal == sig) & (amp.pooling == pool)].set_index("type")
        vals = [a.loc["R", "median_uVpp"], a.loc["R", "IQR_uVpp"], a.loc["FR", "median_uVpp"], a.loc["FR", "IQR_uVpp"]]
        within = [abs(x - e) <= 0.1 * e for x, e in zip(vals, [27.4, 15.0, 9.2, 7.5])]
        print(f"{sig}, {pool}: within tolerance {within}")
# Per patient, reading (a)(i)
per = []
for p in PIDS:
    r = np.concatenate([iv["amp_R"][:, 0] for iv in scans[p]["intervals"]])
    f = np.concatenate([iv["amp_FR"][:, 0] for iv in scans[p]["intervals"]])
    per.append({"patient": p, "R_median": np.nanmedian(r), "FR_median": np.nanmedian(f) if len(f) else np.nan})
print("\nPer patient medians, reading (a)(i):")
print(pd.DataFrame(per).set_index("patient").round(1).T.to_string())

**Result: reproduced.** On the band-passed signal, over all events of all channels, ripples have
median 27.35 µVpp (IQR 14.99) and FR median 9.13 µVpp (IQR 7.37), all four within 10% of 27.4,
15.0, 9.2 and 7.5. Restricted to channels not grayed, 26.53 (14.26) and 8.81 (6.96), also all
within tolerance. On the unfiltered signal the values are far larger (134.04 and 111.13 µVpp), so
the paper's amplitudes are those of the band-passed signal.

## followup-not-different

> Mean follow-up was 29 +- 15 months for patients with poor outcome and 23 +- 11 months for good
> outcome, and the two distributions did not differ (Wilcoxon rank sum test, p = 0.25).

In [ ]:
fu = t1["Postoperative follow-up (months)"].astype(int)
good, poor = fu[ilae == 1], fu[ilae > 1]
print("Follow-up is not in the deposit (no field of the .mat files holds it); the months are Table 1's.")
print(f"Poor outcome: mean {poor.mean():.1f}, SD {poor.std(ddof=1):.1f} months (n {len(poor)})")
print(f"Good outcome: mean {good.mean():.1f}, SD {good.std(ddof=1):.1f} months (n {len(good)})")
pa = stats.mannwhitneyu(poor, good, alternative="two-sided", method="asymptotic", use_continuity=True).pvalue
pe = stats.mannwhitneyu(poor, good, alternative="two-sided", method="exact").pvalue
print(f"Wilcoxon rank sum, normal approximation with continuity correction: p = {pa:.3f}; exact: p = {pe:.3f}")
print(f"Difference of means {poor.mean() - good.mean():.1f} months; median {poor.median()} against {good.median()}")

**Result: reproduced.** The follow-up months are not in the deposit; on Table 1's values the
poor-outcome group has mean 29.3 (SD 15.3) and the good-outcome group 22.8 (SD 10.7) months, all
within tolerance, and the rank sum test gives p = 0.250 (normal approximation with continuity
correction) and 0.275 (exact), above 0.05. This tests the paper's table, not the data set.

## table-s1-rates

> The per-channel rates of ripples, FR and FRandR (events/min) of every patient are those listed in
> Table S1, and the channels marked there as the HFO area of each type are those whose rate exceeds
> the 95th percentile.

In [ ]:
rows = []
for reading in ("a", "b", "c"):
    for nominal in (False, True):
        ok = n = 0
        bad = []
        for p in PIDS:
            C, dur, night = counts[(p, "R")]
            for k in ieeg1.TYPES:
                C, dur, night = counts[(p, k)]
                d = np.full_like(dur, 5.0) if nominal else dur
                r = ieeg1.pooled_rates(C, d, night, reading)
                for c, v in s1[p]["rates"][k].items():
                    if c in labels[p]:
                        n += 1
                        good_ = abs(r[labels[p].index(c)] - v) <= max(0.1 * v, 0.1)
                        ok += good_
                        if not good_:
                            bad.append(p)
        rows.append({"pooling": reading, "durations": "5 min each" if nominal else "actual length",
                     "match": ok, "of": n, "share": ok / n,
                     "mismatches_by_patient": dict(pd.Series(bad).value_counts().sort_index()) if bad else {}})
print(pd.DataFrame(rows).to_string(index=False))
# Patient 10 is printed in whole numbers in Table 1S: the same comparison with 0.5/min for that patient
ok = n = 0
for p in PIDS:
    for k in ieeg1.TYPES:
        r = rates(p, k, "a")
        for c, v in s1[p]["rates"][k].items():
            if c in labels[p]:
                n += 1
                ok += abs(r[labels[p].index(c)] - v) <= max(0.1 * v, 0.5 if p == 10 else 0.1)
print(f"\nVariant, reading (a) with the rounding of patient 10's whole numbers (0.5/min): {ok} of {n} ({ok / n:.1%})")
print("Patient 10 FRandR, Table 1S against reading (a):",
      [(c, v, round(rates(10, 'FRandR')[labels[10].index(c)], 2)) for c, v in list(s1[10]["rates"]["FRandR"].items())[:12]])
# HFO areas against the green cells
arows = []
for reading in ("a", "b", "c"):
    for cs in ("a", "b"):
        for strict in (True, False):
            agree = sum(area(p, k, reading, cs, strict)[0] == s1[p]["area"][k] for p in PIDS for k in ieeg1.TYPES)
            arows.append({"pooling": reading, "channels": cs, "rule": ">" if strict else ">=",
                          "areas_equal_to_green_cells": agree, "of": 60})
print("\nHFO areas (patient x type) equal to Table 1S's green cells:")
print(pd.DataFrame(arows).to_string(index=False))
print("\nAreas, pooling (a), channel set (a), rule '>':")
for p in PIDS:
    print(p, {k: sorted(area(p, k)[0]) for k in ieeg1.TYPES})

**Result: reproduced.** With all events over the total recorded time (reading a), 1766 of 1800
channel and type rates (98.1%) match Table 1S; reading (b) gives 98.1% and (c) 94.7%. The 34
mismatches under (a) are 33 of patient 10, whose rates Table 1S prints as whole numbers, and one of
patient 3; with 0.5/min for patient 10, 1799 of 1800 match. The HFO areas computed with reading (a),
channel set (a) and "greater than" equal the green cells of Table 1S for all 60 patient and type
pairs; with "greater or equal" 56, and with all bipolar channels 48.

## fig1-patient4

> In patient 4 the HFO types differ in their distribution over channels; several channels have
> FRandR rate 0; the ripple area was not resected, and the FRandR area, channel AR1-AR2, was
> resected.

In [ ]:
p = 4
keep = chanset[(p, "a")]
df4 = pd.DataFrame({k: rates(p, k)[keep] for k in ieeg1.TYPES}, index=[labels[p][i] for i in keep])
df4["resected"] = [c in s1[p]["resected"] for c in df4.index]
print(df4.round(2).to_string())
a4 = {k: area(p, k) for k in ieeg1.TYPES}
for k in ieeg1.TYPES:
    print(f"{k}: threshold {a4[k][1]:.2f}/min, area {sorted(a4[k][0])}, outside RA {sorted(a4[k][0] - s1[p]['resected'])}")
print("Channels with FRandR rate 0.0 at one decimal:", int((df4.FRandR.round(1) == 0).sum()))
fig, axs = plt.subplots(1, 3, figsize=(14, 3))
for ax, k in zip(axs, ieeg1.TYPES):
    ax.bar(range(len(df4)), df4[k], color=np.where(df4.resected, "tab:red", "tab:blue"))
    ax.axhline(a4[k][1], color="k")
    ax.set_xticks(range(len(df4)))
    ax.set_xticklabels(df4.index, rotation=90, fontsize=6)
    ax.set_title(f"patient 4, {k} (events/min)")
plt.tight_layout()
plt.show()
print("Plotted rates range from", df4[list(ieeg1.TYPES)].min().min().round(2), "to", df4[list(ieeg1.TYPES)].max().max().round(2))

**Result: reproduced.** In patient 4 the FRandR area is AR1-2 alone (threshold 12.85/min) and
AR1-2 is resected; the ripple area is PL2-3, outside the resected channels; 9 of 24 channels have
FRandR rate 0.0 at one decimal. The ripple and FR areas (PL2-3, AR1-2) differ, as the figure shows.

## frandr-area-resection-per-patient

> The FRandR area was fully inside the resected area in all 13 seizure-free patients and in 3 of the
> 7 patients with recurrent seizures (7, 8, 9), and not fully resected in 4 patients with recurrent
> seizures (17, 18, 19, 20).

In [ ]:
col = {"R": "Ripple area resected?", "FR": "FR area resected?", "FRandR": "FRandR area resected?"}


def resection_table(reading="a", cs="a", strict=True):
    out = []
    for p in PIDS:
        row = {"patient": p, "ILAE": int(ilae[p])}
        for k in ieeg1.TYPES:
            a, _ = area(p, k, reading, cs, strict)
            row[k + "_area"] = ",".join(sorted(a))
            row[k + "_full"] = "Y" if a <= s1[p]["resected"] else "N"
            row[k + "_T1"] = t1.loc[p, col[k]]
        out.append(row)
    return pd.DataFrame(out).set_index("patient")


res_main = resection_table()
print(res_main[["ILAE", "FRandR_area", "FRandR_full", "FRandR_T1"]].to_string())
agree = int((res_main.FRandR_full == res_main.FRandR_T1).sum())
sfm = res_main.ILAE == 1
print(f"Agree with Table 1: {agree} of 20; fully resected among seizure free "
      f"{int((res_main.FRandR_full[sfm] == 'Y').sum())} of {int(sfm.sum())}, among recurrent seizures "
      f"{int((res_main.FRandR_full[~sfm] == 'Y').sum())} of {int((~sfm).sum())}")
print("\nOther readings (pooling x channel set x rule): patients agreeing with Table 1, FRandR")
for reading in ("a", "b", "c"):
    for cs in ("a", "b"):
        for strict in (True, False):
            r = resection_table(reading, cs, strict)
            print(f"  pooling {reading}, channels {cs}, rule {'>' if strict else '>='}: "
                  f"{int((r.FRandR_full == r.FRandR_T1).sum())} of 20, seizure-free fully resected "
                  f"{int((r.FRandR_full[sfm] == 'Y').sum())}, recurrent fully resected {int((r.FRandR_full[~sfm] == 'Y').sum())}")
# Variant (d): a channel is in the area when it is in the per-interval area in at least half the intervals
d_full = []
for p in PIDS:
    C, dur, night = counts[(p, "FRandR")]
    keep = chanset[(p, "a")]
    R = (C / dur[:, None])[:, keep]
    inarea = np.zeros(len(keep))
    for r in R:
        idx, _ = ieeg1.hfo_area(r)
        inarea[idx] += 1
    a = {labels[p][keep[i]] for i in np.flatnonzero(inarea >= len(R) / 2)}
    d_full.append("Y" if a <= s1[p]["resected"] else "N")
print("Variant (d), per-interval areas in at least half the intervals: agree with Table 1 for",
      int(sum(x == y for x, y in zip(d_full, res_main.FRandR_T1))), "of 20")

**Result: reproduced.** The FRandR area is fully inside the resected channels of Table 1S in 13
of 13 seizure-free patients and in 3 of 7 with recurrent seizures (7, 8, 9), and not in 17, 18, 19
and 20: all 20 patients agree with Table 1. The same holds under pooling (b) and (c); with "greater
or equal" 19 of 20 agree, with all bipolar channels 17 of 20, and with per-interval areas held in
at least half the intervals (variant d) 18 of 20.

## ripple-fr-area-resection-per-patient

> The ripple and FR areas were resected in the patients marked in Table 1 (ripple area resected in
> 7 of 13 seizure-free and 4 of 7 recurrent-seizure patients; FR area in 9 of 13 and 5 of 7).

In [ ]:
for k in ("R", "FR"):
    agree = int((res_main[k + "_full"] == res_main[k + "_T1"]).sum())
    print(f"{k}: agree with Table 1 for {agree} of 20; fully resected among seizure free "
          f"{int((res_main[k + '_full'][sfm] == 'Y').sum())}, among recurrent {int((res_main[k + '_full'][~sfm] == 'Y').sum())}")
    dis = res_main[res_main[k + "_full"] != res_main[k + "_T1"]]
    for p, r in dis.iterrows():
        print(f"   patient {p}: area {r[k + '_area']}, resected here {r[k + '_full']}, Table 1 {r[k + '_T1']}, "
              f"Table 1S red channels in the area: {sorted(set(r[k + '_area'].split(',')) & s1[p]['resected'])}, "
              f"Table 1S green cells {sorted(s1[p]['area'][k])}")
print("\nTable 1 counts: ripple area fully resected in", int((t1[col['R']][ilae == 1] == 'Y').sum()), "seizure free and",
      int((t1[col['R']][ilae > 1] == 'Y').sum()), "recurrent; FR area", int((t1[col['FR']][ilae == 1] == 'Y').sum()),
      "and", int((t1[col['FR']][ilae > 1] == 'Y').sum()))
print("\nOther readings: patients agreeing with Table 1 (R, FR)")
for reading in ("a", "b", "c"):
    for cs in ("a", "b"):
        for strict in (True, False):
            r = resection_table(reading, cs, strict)
            print(f"  pooling {reading}, channels {cs}, rule {'>' if strict else '>='}: "
                  f"R {int((r.R_full == r.R_T1).sum())}, FR {int((r.FR_full == r.FR_T1).sum())}; counts R "
                  f"{int((r.R_full[sfm] == 'Y').sum())}/{int((r.R_full[~sfm] == 'Y').sum())}, FR "
                  f"{int((r.FR_full[sfm] == 'Y').sum())}/{int((r.FR_full[~sfm] == 'Y').sum())}")

**Result: partially reproduced.** The ripple area agrees with Table 1 for 17 of 20 patients and
the FR area for 18 of 20. The FR counts match (9 of 13 and 5 of 7); the ripple counts do not (5 of
13 and 3 of 7 against 7 and 4). Every disagreement is between the paper's two tables: in patients
7, 11 and 15 the green ripple cells of Table 1S are not red, yet Table 1 marks the ripple area
resected; in patient 1 the green FR cell is red, yet Table 1 says not resected; in patient 11 one
of the three green FR cells (GR34-35) is not red, yet Table 1 says resected. Here the areas equal
Table 1S's green cells, so the result follows Table 1S.

### The prediction measures

TP, FP, FN, TN, PPV, NPV, sensitivity, specificity and accuracy as in the Methods ("Clinical
validation"); Clopper-Pearson and asymptotic (Wald) 95% intervals; chi2 on the 2 x 2 table of
resection by outcome, (a) Pearson, (b) Yates, (c) Fisher's exact test (claims file).

In [ ]:
def prediction(full, good):
    """TP, FP, FN, TN (Methods, Clinical validation) and the five measures."""
    full, good = np.asarray(full) == "Y", np.asarray(good, bool)
    TP = int(np.sum(~full & ~good))
    FP = int(np.sum(~full & good))
    FN = int(np.sum(full & ~good))
    TN = int(np.sum(full & good))
    m = {"TP": TP, "FP": FP, "FN": FN, "TN": TN}
    pairs = {"specificity": (TN, TN + FP), "sensitivity": (TP, TP + FN), "NPV": (TN, TN + FN),
             "PPV": (TP, TP + FP), "accuracy": (TP + TN, len(full))}
    for name, (k, n) in pairs.items():
        m[name] = 100 * k / n if n else np.nan
        if n:
            ci = stats.binomtest(k, n).proportion_ci(method="exact")
            m[name + "_CP"] = (round(100 * ci.low, 1), round(100 * ci.high, 1))
            ph = k / n
            h = 1.96 * np.sqrt(ph * (1 - ph) / n)
            m[name + "_Wald"] = (round(100 * max(ph - h, 0), 1), round(100 * min(ph + h, 1), 1))
    tab = np.array([[TP, FN], [FP, TN]])
    m["chi2_pearson_p"] = stats.chi2_contingency(tab, correction=False)[1]
    m["chi2_yates_p"] = stats.chi2_contingency(tab, correction=True)[1]
    m["fisher_p"] = stats.fisher_exact(tab)[1]
    return m

## table2-frandr-prediction

> Defining the HFO area by FRandR, its resection predicted seizure outcome with specificity 100%
> (CI 78-100%), sensitivity 57% (CI 18-90%), NPV 81% (CI 54-96%), PPV 100%, and accuracy 85% (CI
> 62-97%, p = 0.002, chi2 test).

In [ ]:
mf = prediction(res_main.FRandR_full, res_main.ILAE == 1)
for k, v in mf.items():
    print(f"{k}: {np.round(v, 4) if isinstance(v, float) else v}")
print("\nRounded percentages:", {k: round(mf[k]) for k in ("specificity", "sensitivity", "NPV", "PPV", "accuracy")})
print("Paper: specificity 100 (78-100), sensitivity 57 (18-90), NPV 81 (54-96), PPV 100, accuracy 85 (62-97), p = 0.002")

**Result: reproduced.** TP 4, FP 0, FN 3, TN 13: specificity 100%, sensitivity 57%, NPV 81%, PPV
100% and accuracy 85%, all equal to the paper. Clopper-Pearson intervals: sensitivity 18.4-90.1,
NPV 54.4-96.0, accuracy 62.1-96.8, as printed; specificity 75.3-100 against the printed 78-100. The
asymptotic intervals do not match the printed ones (sensitivity 20.5-93.8). Chi2 p = 0.0023 without
correction (reading a), 0.0138 with Yates' correction and 0.0072 by Fisher's exact test, all below
0.05.

## table2-ripple-fr-prediction

> With the area defined by ripples, specificity was 54%, sensitivity 43%, NPV 64%, PPV 33% and
> accuracy 50%; by FR, 69%, 29%, 64%, 33% and 55%.

In [ ]:
t2 = {}
for k in ("R", "FR"):
    t2[k] = prediction(res_main[k + "_full"], res_main.ILAE == 1)
    print(k, {m: round(t2[k][m]) for m in ("specificity", "sensitivity", "NPV", "PPV", "accuracy")},
          {m: t2[k][m] for m in ("TP", "FP", "FN", "TN")})
print("Paper, Table 2: ripple 54, 43, 64, 33, 50; FR 69, 29, 64, 33, 55")
print("\nThe same measures from Table 1's own Y/N columns:")
for k in ("R", "FR"):
    m = prediction(t1[col[k]].values, (ilae == 1).values)
    print(k, {x: round(m[x]) for x in ("specificity", "sensitivity", "NPV", "PPV", "accuracy")})
print("\nAcross readings (pooling, channel set, rule), the ripple and FR measures:")
for reading in ("a", "b", "c"):
    for cs in ("a", "b"):
        for strict in (True, False):
            r = resection_table(reading, cs, strict)
            print(f"  {reading},{cs},{'>' if strict else '>='}:",
                  {k: [round(prediction(r[k + '_full'], r.ILAE == 1)[m]) for m in ("specificity", "sensitivity", "NPV", "PPV", "accuracy")]
                   for k in ("R", "FR")})

**Result: partially reproduced.** By FR, specificity 69%, sensitivity 29%, NPV 64%, PPV 33% and
accuracy 55%, all five as in Table 2. By ripples, 38%, 57%, 62%, 33% and 45% against 54, 43, 64, 33
and 50: only PPV matches. Computed from Table 1's own Y/N columns the ripple values are those of
Table 2, so the difference comes from the three patients where Table 1S and Table 1 disagree on the
ripple area (claim ripple-fr-area-resection-per-patient).

## frandr-more-specific

> The FRandR area was more specific in predicting seizure freedom than the ripple and FR areas
> separately (specificity 100% against 54% and 69%).

In [ ]:
print(f"Specificity: FRandR {mf['specificity']:.0f}%, ripples {t2['R']['specificity']:.0f}%, FR {t2['FR']['specificity']:.0f}%")

**Result: reproduced.** Specificity is 100% for the FRandR area against 38% for ripples and 69%
for FR.

## engel-prediction

> Taking Engel class I (ILAE 1-3) as good outcome, the FRandR area gives sensitivity 80% (CI
> 28-99%), NPV 94% (CI 70-100%) and accuracy 95% (CI 75-100%, p = 0.001, chi2 test).

In [ ]:
me = prediction(res_main.FRandR_full, res_main.ILAE <= 3)
print({k: (np.round(v, 4) if isinstance(v, float) else v) for k, v in me.items()})
print("Rounded:", {k: round(me[k]) for k in ("sensitivity", "NPV", "accuracy")})
print("Paper: sensitivity 80 (28-99), NPV 94 (70-100), accuracy 95 (75-100), p = 0.001")

**Result: reproduced.** With ILAE 1-3 as good outcome: TP 4, FP 0, FN 1, TN 15, sensitivity 80%
(Clopper-Pearson 28.4-99.5), NPV 94% (69.8-99.8) and accuracy 95% (75.1-99.9). Chi2 p = 0.0001
(Pearson), 0.0012 (Yates) and 0.0010 (Fisher), all below 0.05; the printed p = 0.001 fits Yates or
Fisher.

## ete-soz-not-resected (blocked)

> In all seizure-free ETE patients (10-16) the resection was limited to 1-2 cm2 and the seizure onset
> zone was not fully resected, while the FRandR area was always fully resected.

Not tested. The seizure onset zone of patients 10 to 16 is in neither the deposit (no field of the
files, nothing in the data description or the electrode schemes) nor the paper. The FRandR part is
judged by frandr-area-resection-per-patient, where the FRandR area of each of patients 10 to 16 is
fully inside the resected channels of Table 1S.

## fig2-patient17

> In patient 17 (recurrent seizures) the resected area does not comprise all channels of the HFO
> area, and the HFO area is smaller than the seizure onset zone.

In [ ]:
p = 17
for strict in (True, False):
    for cs in ("a", "b"):
        a17, thr = area(p, "FRandR", "a", cs, strict)
        print(f"channels {cs}, rule {'>' if strict else '>='}: threshold {thr:.3f}/min, FRandR area {sorted(a17)}, "
              f"inside RA {sorted(a17 & s1[p]['resected'])}, outside RA {sorted(a17 - s1[p]['resected'])}")
r17 = rates(p, "FRandR")
top = np.argsort(r17)[::-1][:6]
print("Highest FRandR rates of patient 17:", [(labels[p][i], round(r17[i], 2), labels[p][i] in s1[p]["resected"]) for i in top])

**Result: partially reproduced.** Patient 17's FRandR area is FPR10-11 alone (threshold
2.00/min), outside the resected channels, so the resection does not comprise the HFO area. No area
channel is inside the resection, which the criterion also asks for. With "greater or equal" the
area is FAR3-4, FPR10-11, FPR9-10 and TR3-4 (all at 2.0/min or above), two inside and two outside
the resection. The comparison with the seizure onset zone cannot be made: no SOZ channels are in
the deposit or the paper.

## fig3-patient1

> In patient 1 (seizure free) the FRandR area was channel PHR1-2 in each of 4 nights and in the
> resected area for all intervals, and 99% of the scalar products between interval pairs exceeded
> significance.

In [ ]:
p = 1
C, dur, night = counts[(p, "FRandR")]
keep = chanset[(p, "a")]
R = (C / dur[:, None])[:, keep]
names = [labels[p][i] for i in keep]
for u in np.unique(night):
    idx, thr = ieeg1.hfo_area(R[night == u].mean(0))
    print(f"night {u}: {int(np.sum(night == u))} intervals, area {[names[i] for i in idx]} (threshold {thr:.2f})")
inside, empty = 0, 0
for r in R:
    idx, _ = ieeg1.hfo_area(r)
    a = {names[i] for i in idx}
    inside += a <= s1[p]["resected"]
    empty += len(a) == 0
print(f"Per-interval areas inside the RA: {inside} of {len(R)} (empty areas: {empty})")
rng = np.random.default_rng(1)
fa = ieeg1.test_retest(R, 5000, rng)
rng = np.random.default_rng(1)
fb = ieeg1.test_retest(R, 5000, rng, pooled_null=True)
print(f"Share of significant interval pairs: null per pair (a) {fa[0]:.3f}, pooled null (b) {fb[0]:.3f} "
      f"({fa[1]} pairs, {fa[2]} without a scalar product)")

**Result: reproduced.** Patient 1's FRandR area is PHR1-2 in each of the 4 nights; the
per-interval areas lie inside the resected channels in 28 of 28 intervals; 99.2% of the 378
interval pairs are significant with a null per pair (97.6% with a pooled null), within 0.099 of
0.99 under both.

## test-retest-group

> Across 18 patients the share of significant scalar products had median 97.5% (lower quartile 67%)
> over pairs of 5-minute intervals and median 100% (lower quartile 100%) over pairs of nights.

Per interval, the rate vector over channel set (a) with each interval's actual length; nights, the
mean of the night's interval rates (Methods, "we took the mean HFO rate for each night"); 5000
permutations; seeds fixed per claim.

In [ ]:
tr_rows = []
for k in ieeg1.TYPES:
    for pooled in (False, True):
        rng = np.random.default_rng(2017)
        for p in PIDS:
            C, dur, night = counts[(p, k)]
            R = (C / dur[:, None])[:, chanset[(p, "a")]]
            fi = ieeg1.test_retest(R, 5000, rng, pooled)
            Rn = np.array([R[night == u].mean(0) for u in np.unique(night)])
            fn = ieeg1.test_retest(Rn, 5000, rng, pooled)
            tr_rows.append({"type": k, "null": "pooled (b)" if pooled else "per pair (a)", "patient": p,
                            "intervals": fi[0], "nights": fn[0], "undefined_pairs": fi[2] + fn[2]})
tr = pd.DataFrame(tr_rows)
t1_int = pd.to_numeric(t1["Test - retest intervals"], errors="coerce")
t1_nig = pd.to_numeric(t1["Test - retest nights"], errors="coerce")
summ = []
for (k, nl), g in tr.groupby(["type", "null"], sort=False):
    g = g.set_index("patient")
    vi, vn = g.intervals.dropna(), g.nights.dropna()
    summ.append({"type": k, "null": nl, "n_int": len(vi), "median_int_%": 100 * ieeg1.prctile_matlab(vi, 50),
                 "lowq_int_%": 100 * ieeg1.prctile_matlab(vi, 25), "n_night": len(vn),
                 "median_night_%": 100 * ieeg1.prctile_matlab(vn, 50), "lowq_night_%": 100 * ieeg1.prctile_matlab(vn, 25),
                 "patients_within_0.1_T1_int": int(np.sum(np.abs(g.intervals - t1_int).dropna() <= 0.1)),
                 "patients_within_0.1_T1_night": int(np.sum(np.abs(g.nights - t1_nig).dropna() <= 0.1)),
                 "undefined_pairs": int(g.undefined_pairs.sum())})
summ = pd.DataFrame(summ)
print(summ.round(1).to_string(index=False))
print("\nPaper: intervals median 97.5% (lower quartile 67%), nights median 100% (lower quartile 100%), tolerance 10%")
wide = tr[tr.null == "per pair (a)"].pivot(index="patient", columns="type", values="intervals")
wide["Table1"] = t1_int
print("\nPer patient share of significant interval pairs, null per pair (a):")
print(wide.round(2).to_string())
widen = tr[tr.null == "per pair (a)"].pivot(index="patient", columns="type", values="nights")
widen["Table1"] = t1_nig
print("\nPer patient share of significant night pairs, null per pair (a):")
print(widen.round(2).to_string())

**Result: partially reproduced.** With FRandR and a null per pair, the median share of
significant interval pairs over 18 patients is 99.5% (paper 97.5) and over nights 100% (paper 100),
and the night lower quartile is 91.7% (paper 100, within 10); the interval lower quartile is 55.0%
against 67% (tolerance 6.7). No reading puts all four within tolerance (pooled null: 93.4, 58.3,
100, 91.7; ripples and FR are further off). Per patient, 17 of 18 FRandR interval values are within
0.1 of Table 1; patient 8 has 0.55 against 0.67, which moves the lower quartile.

## test-retest-by-group

> Over intervals, the median share of significant scalar products was above 98% in TLE patients
> irrespective of outcome, 97% in ETE patients with good outcome and only 36% in ETE patients with
> poor outcome.

In [ ]:
grp = {"TLE": [p for p in PIDS if t1.loc[p, "Epilepsy"] == "TLE"],
       "ETE good (10-16)": [p for p in PIDS if t1.loc[p, "Epilepsy"] == "ETE" and ilae[p] == 1],
       "ETE poor (17-20)": [p for p in PIDS if t1.loc[p, "Epilepsy"] == "ETE" and ilae[p] > 1]}
for (k, nl), g in tr.groupby(["type", "null"], sort=False):
    g = g.set_index("patient").intervals
    out = {name: (round(100 * ieeg1.prctile_matlab(g[ps].dropna(), 50), 1), int(g[ps].notna().sum())) for name, ps in grp.items()}
    tle_by = {o: round(100 * ieeg1.prctile_matlab(g[[p for p in grp["TLE"] if (ilae[p] == 1) == (o == "ILAE 1")]].dropna(), 50), 1)
              for o in ("ILAE 1", "ILAE 2-6")}
    print(k, nl, "median % (patients):", out, "; TLE by outcome:", tle_by)
print("\nPaper: TLE >98% (irrespective of outcome), ETE good 97%, ETE poor 36%")
print("Table 1's own values:", {name: round(100 * ieeg1.prctile_matlab(t1_int[ps].dropna(), 50), 1) for name, ps in grp.items()})

**Result: reproduced.** With FRandR and a null per pair (the reading closest to Table 1), the
median over intervals is 99.5% in the 8 TLE patients with pairs, 100% in the 7 ETE patients with
good outcome and 35.9% in the 3 ETE patients with poor outcome and pairs, all within tolerance, and
the last is the lowest. Split by outcome the TLE medians are 99.5% (ILAE 1) and 77.5% (ILAE 2-6),
as Table 1's own values (patients 8 and 9) also imply.

## spectral-classes

> Of 6182 FRandR events on the highest-rate FRandR channel in each patient's HFO area, 45% had two
> spectral peaks (second peak median 240 Hz), 28% one peak in the HFO range (median 184 Hz) and 27%
> no prominent peak above 100 Hz.

Method from the Methods ("Spectral analysis") and the claims file's readings: per patient the
channel of the FRandR area with the highest FRandR rate; every FRandR event on it, over all
intervals; Stockwell transform (a port of st.m from the data set's detector.zip) of 0.6 s centred
on the event; the spectrum averaged over the FR's duration (in the deposit the FR and the FRandR of
an event share start and end, so the two readings of the centre coincide); four gaussians fitted
by least squares over 80-500 Hz. Readings run: signal raw or high-passed at 80 Hz; |S| (main) or
power; starting values fixed (100, 200, 300, 400 Hz) or at the spectrum's largest local maxima;
peaks as local maxima of the fitted curve or as fitted centres (amplitude at least 10% of the
curve's maximum); the second peak of bimodal events as the highest-frequency peak or the most
prominent peak. Runs about 17 minutes on four workers.

In [ ]:
jobs = []
top_channel = {}
for p in PIDS:
    a, _ = area(p, "FRandR")
    r = rates(p, "FRandR")
    if not a:
        continue
    c = max(a, key=lambda l: r[labels[p].index(l)])
    ci_ = labels[p].index(c)
    top_channel[p] = c
    ev, st0, sn = [], [], []
    for iv in scans[p]["intervals"]:
        sel = np.flatnonzero(iv["FRandR"][:, 0] == ci_)
        ev.append(iv["FRandR"][sel, 1:])
        st0.append(iv["frandr_snip_start"][sel])
        sn.append(iv["frandr_snips"][sel])
    ev, st0, sn = np.concatenate(ev), np.concatenate(st0), np.concatenate(sn)
    for i0 in range(0, len(ev), 500):       # chunks of 500 events, so 4 workers share the load
        jobs.append({"pid": p, "events": ev[i0:i0 + 500], "snip_start": st0[i0:i0 + 500], "snips": sn[i0:i0 + 500]})
print("Channel per patient (highest FRandR rate in the FRandR area):", top_channel)
print("FRandR events on these channels:", sum(len(j["events"]) for j in jobs))
t_sp = time.time()
with get_context("spawn").Pool(4) as pool:
    spec = list(pool.imap(ieeg1.spectral_events, jobs))
print(f"spectral fits done in {(time.time() - t_sp) / 60:.1f} min")
f_axis = spec[0]["f"]
ev_rows = []
for res_ in spec:
    for fits in res_["fits"]:
        row = {"patient": res_["pid"]}
        for key, (pp, okfit) in fits.items():
            row[key + ("ok",)] = okfit
            for peaks_from in ("curve", "centres"):
                cl, pk, amps = ieeg1.classify(f_axis, pp, peaks_from)
                tag = "/".join(key) + "/" + peaks_from
                row[tag] = cl
                if cl == "bimodal":
                    row[tag + "/second_high"] = pk[pk > 200].max()
                    above = pk > 200
                    row[tag + "/second_prominent"] = pk[np.argmax(amps)]
                if cl == "unimodal":
                    row[tag + "/peak"] = pk[pk > 100][0]
        ev_rows.append(row)
evdf = pd.DataFrame(ev_rows)
readings = [c for c in evdf.columns if isinstance(c, str) and c.count("/") == 3]
sp = []
for rd in readings:
    vc = evdf[rd].value_counts(normalize=True)
    sp.append({"reading (signal/measure/start/peaks)": rd, "bimodal_%": 100 * vc.get("bimodal", 0),
               "unimodal_%": 100 * vc.get("unimodal", 0), "none_%": 100 * vc.get("none", 0),
               "second_peak_high_Hz": evdf.get(rd + "/second_high", pd.Series(dtype=float)).median(),
               "second_peak_prominent_Hz": evdf.get(rd + "/second_prominent", pd.Series(dtype=float)).median(),
               "unimodal_peak_Hz": evdf.get(rd + "/peak", pd.Series(dtype=float)).median()})
sp = pd.DataFrame(sp)
print(sp.round(1).to_string(index=False))
nfail = {k: int((~evdf[k]).sum()) for k in evdf.columns if isinstance(k, tuple)}
print("Fits that did not converge, per reading:", nfail)
print("\nPaper: 6182 events; bimodal 45% (second peak 240 Hz), unimodal 28% (184 Hz), none 27%; tolerance 10%")
for _, r in sp.iterrows():
    shares = [abs(r["bimodal_%"] - 45) <= 4.5, abs(r["unimodal_%"] - 28) <= 2.8, abs(r["none_%"] - 27) <= 2.7]
    print(f"  {r.iloc[0]}: shares within tolerance {shares}, bimodal largest {r['bimodal_%'] > max(r['unimodal_%'], r['none_%'])}")
print("\nEvents per patient:", evdf.patient.value_counts().sort_index().to_dict())

**Result: partially reproduced, resting on assumptions.** The channels give 20421 FRandR events,
3.3 times the paper's 6182. Under the main reading (raw signal, |S|, fixed starting values, peaks
of the fitted curve) 57.7% are bimodal, 8.3% unimodal and 34.0% without a peak, against 45, 28 and
27; no reading puts any share within tolerance except the 28.2% without a peak of one reading
(high-passed signal, starting values at the maxima). Bimodal events are the largest class in 9 of
the 10 readings, as in the paper. The unimodal peak (median 175.0 Hz) is within tolerance of 184
Hz; the second peak of bimodal events is not (341.7 Hz as the highest-frequency peak, 111.7 Hz as
the most prominent, against 240). 378 of 20421 fits under the main reading did not converge and are
classified from their last estimate.

## spectral-tle-ete-no-difference

> The spectral classes of FRandR events did not differ significantly between TLE and ETE channels.

In [ ]:
main = "raw/abs/fixed/curve"
evdf["group"] = np.where(evdf.patient <= 9, "TLE", "ETE")
for rd in readings:
    tab = pd.crosstab(evdf[rd], evdf.group)
    chi = stats.chi2_contingency(tab.values)
    shares = (tab / tab.sum()).round(3)
    fr = evdf.assign(b=evdf[rd] == "bimodal").groupby("patient").b.mean()
    w = stats.mannwhitneyu(fr[fr.index <= 9], fr[fr.index > 9]).pvalue
    print(f"{rd}: chi2 = {chi[0]:.1f}, p = {chi[1]:.3g} (events TLE {int(tab['TLE'].sum())}, ETE {int(tab['ETE'].sum())}); "
          f"shares TLE {shares['TLE'].to_dict()}, ETE {shares['ETE'].to_dict()}; rank sum on per-patient bimodal fraction p = {w:.3f}")
fr = evdf.assign(b=evdf[main] == "bimodal").groupby("patient").b.mean()
print("\nPer-patient bimodal fraction, main reading:", fr.round(2).to_dict())

**Result: not reproduced, resting on assumptions.** The chi2 test of spectral class by epilepsy
type over events, the criterion's main test, finds a difference under every reading (main reading
chi2 = 299.3, p = 1e-65, 11153 TLE and 9268 ETE events; bimodal 53.9% in TLE against 62.2% in ETE).
The variant on per-patient bimodal fractions (9 TLE against 11 ETE channels) finds none (rank sum p
= 0.362 under the main reading, 0.149 to 0.447 under the others except one at 0.023). The
classification itself does not reproduce the paper's shares (claim spectral-classes).

## tp-patients-hfo-area-location (blocked)

> In the patients whose HFO area was not fully resected, the area lay outside the resection; in
> patient 18 over occipital contacts ..., in patient 19 over the parietal and not the resected
> temporal cavernoma, and in patient 20 at the anterior margin ...

Blocked. The FRandR area channels of patients 18 to 20 and their place outside the resected
channels are printed below, but the region of a contact cannot be read: the electrode schemes
(docs/electrode_schemes.pdf) draw the strips and grids on a brain without naming the region of each
contact, and the grid of patient 20 has no contact numbers.

In [ ]:
for p in (18, 19, 20):
    a, _ = area(p, "FRandR")
    print(p, "FRandR area", sorted(a), "outside RA", sorted(a - s1[p]["resected"]),
          "; resected channels (Table 1S)", sorted(s1[p]["resected"]))
print("Electrode name prefixes:", {p: pref[p] for p in (18, 19, 20)})
print(f"\nNotebook run time: {(time.time() - started) / 60:.1f} min")

**Result: blocked (metadata insufficient).** The FRandR areas of patients 18 (PLL2-3), 19
(PL22-23, PL23-24) and 20 (OTL10-11) are outside the resected channels of Table 1S, but whether
they lie over occipital, parietal or anterior cortex cannot be read from the deposit.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| table-s1-link (link) | with inputs (Tables 1 and 1S) | reproduced |
| cohort | dataset, outcome from Table 1 | partially reproduced |
| table1-nights-intervals | dataset | reproduced |
| hfo-amplitudes | dataset | reproduced |
| followup-not-different | Table 1 only | reproduced |
| table-s1-rates | dataset | reproduced |
| fig1-patient4 | dataset, resection from Table 1S | reproduced |
| frandr-area-resection-per-patient | dataset, resection from Table 1S | reproduced |
| ripple-fr-area-resection-per-patient | dataset, resection from Table 1S | partially reproduced |
| table2-frandr-prediction | dataset, resection and outcome from the paper | reproduced |
| table2-ripple-fr-prediction | dataset, resection and outcome from the paper | partially reproduced |
| frandr-more-specific | dataset, resection and outcome from the paper | reproduced |
| engel-prediction | dataset, resection and outcome from the paper | reproduced |
| ete-soz-not-resected | | blocked: data missing |
| fig2-patient17 | dataset, resection from Table 1S | partially reproduced |
| fig3-patient1 | dataset, resection from Table 1S | reproduced |
| test-retest-group | dataset | partially reproduced |
| test-retest-by-group | dataset | reproduced |
| spectral-classes | dataset | partially reproduced (assumed) |
| spectral-tle-ete-no-difference | dataset | not reproduced (assumed) |
| tp-patients-hfo-area-location | | blocked: metadata insufficient |

How the analysis differs from the paper's. The HFO events are the deposit's own detections; the
detector was not rerun. Rates use each interval's actual length, as the intervals are not all 5
minutes. The resected channels and the outcomes are the paper's (Table 1S and Table 1), as the
deposit holds neither. The HFO area uses the channels of Table 1S that are not grayed. The
test-retest null, the HFO type of Table 1's test-retest values and the details of the spectral
fit are not given by the paper; every reading the claims file names was run. The 5000 permutations
use fixed seeds.

What was found about the data set.

- It holds the iEEG, the bipolar montage and the HFO detections of all 20 patients, with the nights
  and intervals of Table 1, and the channel rates of Table 1S follow from it (98.1% of values).
- It holds no seizure outcome, follow-up, epilepsy type, resected channels or seizure onset zone,
  although its title names the outcome of surgery; the outcome claims rest on the paper's tables.
- The data description says each file holds 5 minutes; 93 of 385 intervals are 204 to 720 s long.
- The unit of the signal is not stated; the values are consistent with microvolts.
- 745 events end after the last sample of their interval.
- The electrode schemes do not name the region of each contact.
- The spectral analysis does not reproduce: the selected channels hold 20421 FRandR events, not
  6182, and the spectral shares differ from the paper's under every reading run.
- In the paper, Table 1S's grayed channels listed under patient 11 are patient 14's; Table 1S and
  Table 1 disagree on whether the ripple or FR area was resected in patients 1, 7, 11 and 15; the
  Methods' cap of six intervals per night is exceeded in 14 patients.